**Autor:** Andrés Alejandro Rodríguez Lozano  
**Portafolio:** Portafolio de Andrés Alejandro Rodríguez Lozano

# Capítulo 8 — Matriz de correlación

$$\rho_{ij} = \frac{\mathrm{Cov}(R_i,R_j)}{\sigma_i\sigma_j}$$


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%config InlineBackend.figure_formats = ['svg']
plt.rcParams.update({
    'figure.dpi': 72,
    'savefig.dpi': 72,
    'figure.figsize': (8, 3.8),
    'font.size': 9,
    'svg.fonttype': 'none',
    'path.simplify': True,
    'path.simplify_threshold': 0.2,
})

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT / 'src'))
import portfolio_utils as pu

def load_monthly():
    """Carga precios mensuales: cache mensual, diario, o yfinance."""
    mpath = ROOT / 'data' / 'precios_mensuales.csv'
    if mpath.exists():
        print('Precios mensuales (cache):', mpath.name)
        return pd.read_csv(mpath, index_col=0, parse_dates=True).loc[: '2026-09-30']
    csv = ROOT / 'data' / 'precios_ajustados_diarios.csv'
    if csv.exists():
        daily = pd.read_csv(csv, index_col=0, parse_dates=True)
        print('Precios diarios (cache):', csv.name)
        return pu.to_month_end(daily).loc[: '2026-09-30']
    tickers = sorted(set(list(pu.CURRENT_WEIGHTS) + [pu.BENCHMARK]))
    print('Descargando precios con yfinance…')
    daily = pu.download_prices(tickers)
    return pu.to_month_end(daily).loc[: '2026-09-30']


import seaborn as sns
monthly = load_monthly()
rets = monthly.pct_change().loc['2016-01-31':'2026-09-30']
tickers = list(pu.CURRENT_WEIGHTS)
corr = rets[tickers].corr()
print(corr.round(2))


In [ ]:
# Heatmap de correlación — seaborn/matplotlib
fig, ax = plt.subplots(figsize=(5.5, 4.8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', vmin=-1, vmax=1, square=True, ax=ax, annot_kws={'size': 8})
ax.set_title('Correlación mensual')
plt.tight_layout()
plt.show()


In [ ]:
pairs = []
for i, a in enumerate(tickers):
    for b in tickers[i+1:]:
        pairs.append((a, b, float(corr.loc[a, b])))
pairs = sorted(pairs, key=lambda x: x[2], reverse=True)
print('Más correlacionados:')
for a, b, r in pairs[:3]:
    print(f'  {a}-{b}: {r:.3f}')
print('Menos correlacionados:')
for a, b, r in pairs[-3:]:
    print(f'  {a}-{b}: {r:.3f}')


## Conclusiones
1. Diversificación real del núcleo viene sobre todo de IEMG y BRK-B.

## Ejercicios
1. Correlaciones en ventanas de 36 meses.
2. Añade TLT hipotéticamente.
3. Compara 2020 vs 2022.
